# Fase 1.1 — Auditoría de las tablas del export de FINTECH

**Tesis v2 · Eider Johan Díaz Berrío · MCDA EAFIT**

Antes de reconstruir la mora o de pensar en modelos, quiero saber si puedo confiar en la base. En este notebook reviso las 15 tablas del export del 2026-10-08 con las preguntas de siempre en una auditoría de datos:

1. **Inventario:** qué hay, cuántas filas y columnas, y qué columnas personales descarto al leer.
2. **Llaves y duplicados:** si cada tabla tiene una llave única y si hay registros repetidos.
3. **Integridad referencial:** si las relaciones entre tablas se sostienen (por ejemplo, que cada pago apunte a un crédito que existe).
4. **Consistencia de fechas y valores:** que el orden solicitud → inicio → desembolso → pagos tenga sentido y que los montos, plazos y tasas estén en rangos plausibles.

Cada regla termina con un número y una acción, y todo queda en `resultados/fase1/reglas_calidad.csv`. Aquí no muestro ninguna fila de clientes: solo conteos y distribuciones agregadas.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
import warnings

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from tesis_v2 import config, io_datos
from tesis_v2.io_datos import fecha, llave, numero

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160, "display.max_columns", 30, "display.max_rows", 80)
config.asegurar_carpetas()
reglas = []          # (código, regla, afectados, base, acción)


def regla(codigo, descripcion, afectados, base, accion):
    reglas.append({"codigo": codigo, "regla": descripcion, "afectados": int(afectados), "base": int(base),
                   "pct": round(100 * afectados / base, 2) if base else np.nan, "accion": accion})

## 1. Inventario

La función `leer_tabla` descarta las columnas personales (nombres, documento, correo, teléfono, dirección, cuentas, fecha de nacimiento) **antes** de cargarlas en memoria. Así, aunque me equivoque más adelante, esos datos no pueden terminar en una salida del notebook.

In [2]:
NOMBRES = ["credito", "solicitud", "cliente", "empresas", "desembolso", "proyeccionpagos", "transaccionpago",
           "saldo", "saldomesames", "pagosempresas", "facturacioninteres", "avales", "consultas",
           "tipocredito", "gastosadministrativos"]
T, inv = {}, []
for n in NOMBRES:
    path = config.EXPORT_FINTECH / f"bd_{n}.parquet"
    todas = pq.read_schema(path).names
    T[n] = io_datos.leer_tabla(n)
    inv.append({"tabla": f"bd_{n}", "filas": len(T[n]), "columnas_totales": len(todas),
                "columnas_personales_descartadas": len(set(todas) & config.COLUMNAS_PII),
                "llave_id_unica": bool(T[n]["id"].is_unique) if "id" in T[n] else None,
                "id_nulos": int(T[n]["id"].isna().sum()) if "id" in T[n] else None})
inventario = pd.DataFrame(inv)
io_datos.guardar_resultado(inventario, "fase1", "inventario_tablas")
inventario

,tabla,filas,columnas_totales,columnas_personales_descartadas,llave_id_unica,id_nulos
0,bd_credito,1267,34,1,True,0
1,bd_solicitud,1335,34,1,True,0
2,bd_cliente,670,17,10,True,0
3,bd_empresas,47,10,5,True,0
4,bd_desembolso,1228,8,1,True,0
5,bd_proyeccionpagos,70215,17,0,True,0
6,bd_transaccionpago,9964,9,1,True,0
7,bd_saldo,1226,24,0,True,0
8,bd_saldomesames,10795,11,0,True,0
9,bd_pagosempresas,6118,13,0,True,0


Las 15 tablas tienen una columna `id` sin nulos y única. Es la condición mínima para poder hacer cruces sin multiplicar filas. Las columnas personales descartadas se concentran en `bd_cliente`, `bd_empresas`, `bd_desembolso` y `bd_consultas`.

In [3]:
for _, r in inventario.iterrows():
    regla("R01", f"Llave id única en {r['tabla']}", 0 if r["llave_id_unica"] else 1, 1, "Ninguna")

## 2. Duplicados

Un duplicado exacto (todas las columnas iguales salvo el `id`) suele ser un registro cargado dos veces. En los pagos esto importa mucho: un pago duplicado reduce artificialmente la mora. Reviso también los **casi duplicados** de pagos: mismo crédito, misma fecha y mismo valor, aunque difieran en la fuente o la hora de registro.

In [4]:
dup = []
for n, df in T.items():
    cols = [c for c in df.columns if c != "id"]
    dup.append({"tabla": f"bd_{n}", "filas": len(df), "duplicados_exactos_sin_id": int(df.duplicated(cols).sum())})
dup = pd.DataFrame(dup)
dup.loc[dup["tabla"].isin(["bd_cliente", "bd_empresas"]), "duplicados_exactos_sin_id"] = np.nan
io_datos.guardar_resultado(dup, "fase1", "duplicados_por_tabla")
dup[dup["duplicados_exactos_sin_id"].fillna(0) > 0]

,tabla,filas,duplicados_exactos_sin_id
0,bd_credito,1267,5.0
1,bd_solicitud,1335,2.0
6,bd_transaccionpago,9964,1.0
9,bd_pagosempresas,6118,13.0
10,bd_facturacioninteres,8562,10.0


En `bd_cliente` y `bd_empresas` dejo el conteo vacío: al descartar las columnas personales, casi todas las filas quedan "iguales" y el número no significa nada. La pregunta relevante allí es otra: **¿hay una misma persona registrada con dos `id` de cliente?** Si la hay, la validación agrupada por cliente fallaría, porque créditos de la misma persona podrían caer en pliegues distintos. Lo verifico comparando el documento de identidad en memoria, sin mostrarlo. Solo sale el conteo.

In [5]:
_doc = io_datos.leer_parquet(config.EXPORT_FINTECH / "bd_cliente.parquet", ["id", "identificacion"])
_k = _doc["identificacion"].astype("string").str.replace(r"\D", "", regex=True)
clientes_dup = int(_k.duplicated(keep=False).sum())
personas_dup = int(_k[_k.duplicated(keep=False)].nunique())
# Grupo de cliente unificado: el menor id entre los registros que comparten documento.
_g = pd.DataFrame({"cli": llave(_doc["id"]), "doc": _k.fillna("SIN_DOC_" + _doc["id"].astype(str))})
_g["grupo_cliente"] = _g.groupby("doc")["cli"].transform("min")
io_datos.guardar_privado(_g[["cli", "grupo_cliente"]], "grupo_cliente_v2")
del _doc, _k, _g
print(f"Registros de cliente que comparten documento con otro registro: {clientes_dup} ({personas_dup} personas)")
regla("R01b", "Misma persona con más de un id de cliente", clientes_dup, len(T["cliente"]),
      "Ninguna" if clientes_dup == 0 else "Unificar el grupo de cliente para la validación agrupada")

Registros de cliente que comparten documento con otro registro: 2 (1 personas)


In [6]:
cred_cols = [c for c in T["credito"].columns if c != "id"]
mask_dup = T["credito"].duplicated(cred_cols, keep=False)
print("Créditos duplicados exactos (sin id) por estado:", T["credito"].loc[mask_dup, "estado"].value_counts().to_dict())
sol_cols = [c for c in T["solicitud"].columns if c != "id"]
mask_sdup = T["solicitud"].duplicated(sol_cols, keep=False)
print("Solicitudes duplicadas exactas (sin id) por estado:", T["solicitud"].loc[mask_sdup, "estado"].value_counts().to_dict())
n_dup_vivos = int(T["credito"].loc[mask_dup & ~T["credito"]["estado"].isin(["ANULADO", "PENDIENTE_DESEMBOLSO"])]
                  .duplicated(cred_cols).sum())
regla("R01c", "Créditos no anulados duplicados exactos (sin id)", n_dup_vivos, 1161,
      "Ninguna si es 0 (los duplicados están en anulados)")

Créditos duplicados exactos (sin id) por estado: {'ANULADO': 9}
Solicitudes duplicadas exactas (sin id) por estado: {'RECHAZADA': 3}


In [7]:
tp = T["transaccionpago"].assign(valor_n=numero(T["transaccionpago"]["valor"]))
act = tp[(tp["estado"] == "ACTIVO") & (tp["valor_n"] > 0)]
exactos = act.duplicated(["credito_id", "fechapago", "valor", "fuente", "fechalog"]).sum()
casi = act.duplicated(["credito_id", "fechapago", "valor"]).sum()
print(f"Pagos ACTIVO con valor > 0: {len(act):,}")
print(f"  duplicados exactos (crédito, fecha, valor, fuente, fecha de registro): {exactos}")
print(f"  casi duplicados (mismo crédito, fecha y valor): {casi}")
print(f"Pagos ANULADO: {(tp['estado'] == 'ANULADO').sum():,} ({100 * (tp['estado'] == 'ANULADO').mean():.1f} %)")
regla("R02", "Pagos ACTIVO duplicados exactos", exactos, len(act), "Se eliminan antes de reconstruir la mora")
regla("R03", "Pagos ACTIVO casi duplicados (mismo crédito, fecha y valor)", casi, len(act),
      "Se conservan (pueden ser dos cuotas legítimas el mismo día); se mide su efecto en la Fase 1.2")

Pagos ACTIVO con valor > 0: 8,883
  duplicados exactos (crédito, fecha, valor, fuente, fecha de registro): 0
  casi duplicados (mismo crédito, fecha y valor): 7
Pagos ANULADO: 1,052 (10.6 %)


**Lectura.** Los casi duplicados no se eliminan de forma automática. En libranza es normal que el empleador gire dos descuentos iguales el mismo día, por ejemplo la quincena atrasada y la actual. Borrarlos crearía mora ficticia. Si alguno es un error de carga, el efecto sería subestimar la mora, nunca inventarla.

## 3. Integridad referencial

Verifico que cada llave foránea apunte a un registro existente. Las relaciones críticas para el objetivo son crédito → plan de pagos, crédito → pagos y crédito → desembolso.

In [8]:
ids = {n: set(llave(T[n]["id"]).dropna()) for n in T}
rel = [("credito", "cliente_id", "cliente"), ("credito", "empresa_id", "empresas"),
       ("credito", "solicitud_id", "solicitud"), ("credito", "tipocredito_id", "tipocredito"),
       ("solicitud", "credito_id", "credito"), ("solicitud", "cliente_id", "cliente"),
       ("desembolso", "credito_id", "credito"), ("proyeccionpagos", "credito_id", "credito"),
       ("transaccionpago", "credito_id", "credito"), ("saldo", "credito_id", "credito"),
       ("saldomesames", "credito_id", "credito"), ("pagosempresas", "credito_id", "credito"),
       ("facturacioninteres", "credito_id", "credito"), ("avales", "credito_id", "credito")]
filas = []
for hija, col, madre in rel:
    k = llave(T[hija][col])
    filas.append({"relacion": f"bd_{hija}.{col} → bd_{madre}.id", "filas": len(k), "nulos": int(k.isna().sum()),
                  "huerfanos": int((k.notna() & ~k.isin(ids[madre])).sum())})
integ = pd.DataFrame(filas)
io_datos.guardar_resultado(integ, "fase1", "integridad_referencial")
for r in integ.itertuples():
    regla("R04", f"Huérfanos en {r.relacion}", r.huerfanos, r.filas - r.nulos,
          "Ninguna" if r.huerfanos == 0 else "Se excluyen del cruce y se reportan")
integ

,relacion,filas,nulos,huerfanos
0,bd_credito.cliente_id → bd_cliente.id,1267,0,0
1,bd_credito.empresa_id → bd_empresas.id,1267,46,0
2,bd_credito.solicitud_id → bd_solicitud.id,1267,259,0
3,bd_credito.tipocredito_id → bd_tipocredito.id,1267,0,0
4,bd_solicitud.credito_id → bd_credito.id,1335,1102,0
5,bd_solicitud.cliente_id → bd_cliente.id,1335,0,0
6,bd_desembolso.credito_id → bd_credito.id,1228,0,0
7,bd_proyeccionpagos.credito_id → bd_credito.id,70215,19638,0
8,bd_transaccionpago.credito_id → bd_credito.id,9964,0,0
9,bd_saldo.credito_id → bd_credito.id,1226,0,0


Reviso además la cardinalidad de las relaciones que el objetivo supone 1 a 1: una solicitud por crédito y un desembolso por crédito.

In [9]:
c = T["credito"].assign(cid=llave(T["credito"]["id"]), sid=llave(T["credito"]["solicitud_id"]))
vivos = c[~c["estado"].isin(["ANULADO", "PENDIENTE_DESEMBOLSO"])]
sid_rep = vivos["sid"].dropna().duplicated().sum()
sin_sol = vivos["sid"].isna().sum()
des = T["desembolso"].assign(cid=llave(T["desembolso"]["credito_id"]))
n_des = des.groupby("cid").size()
multi_des = int((n_des > 1).sum())
sin_des = int((~vivos["cid"].isin(n_des.index)).sum())
pr = T["proyeccionpagos"]
con_plan = set(llave(pr.loc[pr["tipoproyeccion"].isin(["DESEMBOLSO", "CREDITO"]), "credito_id"]).dropna())
sin_plan = int((~vivos["cid"].isin(con_plan)).sum())
print(f"Créditos no anulados: {len(vivos)}")
print(f"  sin solicitud enlazada: {sin_sol} | solicitud compartida por más de un crédito: {sid_rep}")
print(f"  sin desembolso registrado: {sin_des} | con más de un desembolso: {multi_des}")
print(f"  sin plan de pagos (DESEMBOLSO o CREDITO): {sin_plan}")
regla("R05", "Crédito no anulado sin solicitud enlazada", sin_sol, len(vivos), "t0 = fecha inicial del crédito")
regla("R06", "Solicitud enlazada a más de un crédito", sid_rep, len(vivos), "Revisar; no afecta el objetivo")
regla("R07", "Crédito no anulado sin desembolso registrado", sin_des, len(vivos),
      "t_ref = fecha inicial del crédito")
regla("R08", "Crédito con más de un desembolso", multi_des, len(vivos), "Se toma la primera fecha de desembolso")
regla("R09", "Crédito no anulado sin plan de pagos", sin_plan, len(vivos), "No se puede reconstruir la mora")

Créditos no anulados: 1161
  sin solicitud enlazada: 194 | solicitud compartida por más de un crédito: 0
  sin desembolso registrado: 0 | con más de un desembolso: 0
  sin plan de pagos (DESEMBOLSO o CREDITO): 0


## 4. Consistencia de fechas

Primero miro si las fechas se pueden leer y en qué rango caen. Después verifico el orden lógico del ciclo del crédito. La fecha de referencia t0 (solicitud) define qué información es "pasado" para el modelo, y la fecha de desembolso t_ref define cuándo empieza la ventana de 12 meses.

In [10]:
cols_fecha = [("credito", "fechainicialcredito"), ("credito", "fechavencimientocredito"),
              ("credito", "fechaposibledesembolso"), ("solicitud", "fechasolicitud"),
              ("desembolso", "fechadesembolso"), ("proyeccionpagos", "fecha"),
              ("transaccionpago", "fechapago"), ("saldo", "fecha_transaccion"),
              ("saldomesames", "fecha_cierre_mes"), ("pagosempresas", "fechapago")]
fx = []
for n, col in cols_fecha:
    s = T[n][col]
    f = fecha(s)
    fx.append({"tabla.columna": f"bd_{n}.{col}", "no_nulos": int(s.notna().sum()),
               "no_parseables": int((s.notna() & f.isna()).sum()),
               "minima": f.min().date() if f.notna().any() else None,
               "maxima": f.max().date() if f.notna().any() else None})
fx = pd.DataFrame(fx)
io_datos.guardar_resultado(fx, "fase1", "rango_fechas")
fx

,tabla.columna,no_nulos,no_parseables,minima,maxima
0,bd_credito.fechainicialcredito,1267,0,2021-07-19,2026-10-08
1,bd_credito.fechavencimientocredito,1267,0,2022-02-23,2044-10-04
2,bd_credito.fechaposibledesembolso,1267,0,2021-07-23,2026-10-08
3,bd_solicitud.fechasolicitud,1335,0,2023-05-17,2026-10-08
4,bd_desembolso.fechadesembolso,1228,0,2021-07-23,2026-10-08
5,bd_proyeccionpagos.fecha,70214,0,2021-08-23,2044-10-04
6,bd_transaccionpago.fechapago,9964,0,2021-08-06,2026-10-08
7,bd_saldo.fecha_transaccion,1226,0,2023-02-17,2026-10-08
8,bd_saldomesames.fecha_cierre_mes,10795,0,2021-09-30,2026-10-31
9,bd_pagosempresas.fechapago,6118,0,2021-10-10,2025-09-04


In [11]:
CORTE = pd.Timestamp("2026-10-08")
s = T["solicitud"].assign(sid=llave(T["solicitud"]["id"]), fs=fecha(T["solicitud"]["fechasolicitud"]))
v = vivos.merge(s[["sid", "fs"]], on="sid", how="left")
v["fi"], v["fv"] = fecha(v["fechainicialcredito"]), fecha(v["fechavencimientocredito"])
v["fd"] = v["cid"].map(des.assign(f=fecha(des["fechadesembolso"])).groupby("cid")["f"].min())
dif_sol_ini = (v["fi"] - v["fs"]).dt.days
dif_ini_des = (v["fd"] - v["fi"]).dt.days

chequeos = {
    "R10 Solicitud posterior al inicio del crédito": (dif_sol_ini < 0, dif_sol_ini.notna()),
    "R11 Solicitud más de 90 días antes del inicio": (dif_sol_ini > 90, dif_sol_ini.notna()),
    "R12 Desembolso antes del inicio del crédito": (dif_ini_des < 0, dif_ini_des.notna()),
    "R13 Desembolso más de 30 días después del inicio": (dif_ini_des > 30, dif_ini_des.notna()),
    "R14 Vencimiento contractual anterior o igual al inicio": (v["fv"] <= v["fi"], v["fv"].notna()),
    "R15 Inicio del crédito posterior al corte": (v["fi"] > CORTE, v["fi"].notna()),
}
for k, (mal, base) in chequeos.items():
    cod, txt = k.split(" ", 1)
    regla(cod, txt, (mal & base).sum(), base.sum(),
          "t0 = mínimo entre solicitud e inicio" if cod in ("R10", "R11") else
          "t_ref = primera fecha de desembolso" if cod in ("R12", "R13") else "Revisar con la operación")
pd.DataFrame(reglas[-6:])[["codigo", "regla", "afectados", "base", "pct"]]

,codigo,regla,afectados,base,pct
0,R10,Solicitud posterior al inicio del crédito,26,967,2.69
1,R11,Solicitud más de 90 días antes del inicio,0,967,0.00
2,R12,Desembolso antes del inicio del crédito,0,1161,0.00
3,R13,Desembolso más de 30 días después del inicio,2,1161,0.17
4,R14,Vencimiento contractual anterior o igual al in...,1,1161,0.09
5,R15,Inicio del crédito posterior al corte,0,1161,0.00


In [12]:
print("Días entre solicitud e inicio del crédito (percentiles):")
print(dif_sol_ini.describe(percentiles=[.01, .05, .5, .95, .99]).round(1).to_string())
print("\nDías entre inicio y desembolso (percentiles):")
print(dif_ini_des.describe(percentiles=[.01, .05, .5, .95, .99]).round(1).to_string())

Días entre solicitud e inicio del crédito (percentiles):
count    967.0
mean       1.6
std        2.8
min       -7.0
1%        -1.0
5%         0.0
50%        0.0
95%        7.0
99%       11.3
max       31.0

Días entre inicio y desembolso (percentiles):
count    1161.0
mean        1.0
std         3.2
min         0.0
1%          0.0
5%          0.0
50%         0.0
95%         6.0
99%        15.0
max        45.0


**Lectura.** El ciclo es corto y ordenado. La mediana entre solicitud e inicio es de 0 días y el máximo es de un mes; el desembolso ocurre en mediana el mismo día del inicio. Hay pocos casos con la solicitud registrada unos días *después* del inicio (2,7 % de los créditos con solicitud), y ningún desembolso anterior al inicio. Además, `bd_solicitud` empieza en mayo de 2023: los 194 créditos anteriores no tienen solicitud enlazada y su t0 es la fecha inicial del crédito. Por todo esto mantengo la regla de la v1: **t0 = la fecha más temprana entre la solicitud y el inicio**. Es la elección conservadora para no usar como "pasado" información que llegó después.

Ahora reviso el orden de los pagos y del plan frente al desembolso.

In [13]:
pg = act.assign(cid=llave(act["credito_id"]), f=fecha(act["fechapago"]))
pg = pg.merge(v[["cid", "fd", "fi"]], on="cid", how="inner")
pg["ref"] = pg["fd"].fillna(pg["fi"])
antes = (pg["f"] < pg["ref"] - pd.Timedelta(days=0)).sum()
despues = (pg["f"] > CORTE).sum()
regla("R16", "Pago ACTIVO anterior al desembolso", antes, len(pg),
      "Se conserva: el FIFO lo aplica a la primera cuota (no crea mora)")
regla("R17", "Pago ACTIVO posterior a la fecha de corte", despues, len(pg), "Ninguna si es 0")

plan = pr[pr["tipoproyeccion"] == "DESEMBOLSO"].assign(cid=llave(pr["credito_id"]), f=fecha(pr["fecha"]),
                                                        nc=numero(pr["ncuota"]))
plan = plan[plan["nc"] > 0]
p1 = plan.groupby("cid")["f"].min().rename("primera").to_frame().join(v.set_index("cid")[["fd", "fi"]], how="inner")
p1["ref"] = p1["fd"].fillna(p1["fi"])
regla("R18", "Primera cuota del plan antes del desembolso", (p1["primera"] < p1["ref"]).sum(), len(p1),
      "Revisar; el DPD empieza a contar desde el desembolso")
pd.DataFrame(reglas[-3:])[["codigo", "regla", "afectados", "base", "pct"]]

,codigo,regla,afectados,base,pct
0,R16,Pago ACTIVO anterior al desembolso,0,8883,0.0
1,R17,Pago ACTIVO posterior a la fecha de corte,0,8883,0.0
2,R18,Primera cuota del plan antes del desembolso,0,1160,0.0


## 5. Consistencia de valores

Comparo los términos del crédito con su plan de pagos: número de cuotas pactado frente a cuotas en el plan, y suma del plan frente a la cuota pactada multiplicada por el número de cuotas. También verifico rangos de monto, tasa y edad. La fecha de nacimiento se lee aparte y se descarta en la misma celda: solo sobrevive la edad.

In [14]:
n_cuotas_plan = plan.groupby("cid")["nc"].nunique()
vv = v.set_index("cid")
vv["n_plan"] = n_cuotas_plan
vv["valortotal_n"], vv["cuota_n"], vv["tasa_n"] = numero(vv["valortotal"]), numero(vv["cuotapactada"]), numero(vv["tasa"])
dif_cuotas = (vv["n_plan"] - vv["cantidadcuotas"]).dropna()
regla("R19", "Cuotas del plan DESEMBOLSO distintas de cantidadcuotas", (dif_cuotas != 0).sum(), len(dif_cuotas),
      "Se usa el plan (fechas reales de exigibilidad)")
regla("R20", "Valor total <= 0 o faltante", ((vv["valortotal_n"] <= 0) | vv["valortotal_n"].isna()).sum(), len(vv),
      "Revisar")
print("Tasa pactada (percentiles):", vv["tasa_n"].quantile([0, .01, .5, .99, 1]).round(3).to_dict())
print("Valor total (percentiles, millones COP):", (vv["valortotal_n"].quantile([0, .01, .5, .99, 1]) / 1e6).round(2).to_dict())
print("Cantidad de cuotas (percentiles):", vv["cantidadcuotas"].quantile([0, .01, .5, .99, 1]).to_dict())
print("Diferencia cuotas plan - pactadas:", dif_cuotas.describe().round(2).to_dict())

Tasa pactada (percentiles): {0.0: 0.0, 0.01: 1.35, 0.5: 1.78, 0.99: 2.78, 1.0: 3.0}
Valor total (percentiles, millones COP): {0.0: 0.2, 0.01: 0.38, 0.5: 2.69, 0.99: 42.55, 1.0: 135.23}
Cantidad de cuotas (percentiles): {0.0: 1.0, 0.01: 1.0, 0.5: 14.0, 0.99: 56.40000000000009, 1.0: 270.0}
Diferencia cuotas plan - pactadas: {'count': 1160.0, 'mean': -0.0, 'std': 0.03, 'min': -1.0, '25%': 0.0, '50%': 0.0, '75%': 0.0, 'max': 0.0}


In [15]:
nac = io_datos.leer_parquet(config.EXPORT_FINTECH / "bd_cliente.parquet", ["id", "fechanacimiento"])
edad_cli = pd.Series(fecha(nac["fechanacimiento"]).values, index=llave(nac["id"]))
del nac
vv["edad_t0"] = (vv[["fs", "fi"]].min(axis=1) - vv["cliente_id"].map(edad_cli).values).dt.days / 365.25
del edad_cli
print("Edad en t0 (percentiles):", vv["edad_t0"].quantile([0, .01, .5, .99, 1]).round(1).to_dict())
regla("R21", "Edad en t0 faltante", vv["edad_t0"].isna().sum(), len(vv), "Se deja faltante")
regla("R22", "Edad en t0 < 18 o > 80 años", ((vv["edad_t0"] < 18) | (vv["edad_t0"] > 80)).sum(), len(vv),
      "Se marca como faltante en el dataset analítico")

Edad en t0 (percentiles): {0.0: 18.7, 0.01: 20.3, 0.5: 37.0, 0.99: 69.4, 1.0: 96.2}


## 6. Variables que cambian después del otorgamiento

Algunas columnas de `bd_credito` describen el estado **actual** del crédito, no el del día de la solicitud. Usarlas como predictores sería una fuga de información. Lo verifico empíricamente: `categoria_credito` y `en_refinanciacion` deberían estar asociadas con haber sido refinanciado *después*, y `categoria_inicial` debería ser constante si de verdad es "inicial".

In [16]:
print(pd.crosstab(c["categoria_credito"], c["en_refinanciacion"]).to_string())
print("\ncategoria_inicial:", c["categoria_inicial"].value_counts().to_dict())
refinanciados_despues = set(llave(T["solicitud"].loc[T["solicitud"]["tipo_solicitud"].isin(
    ["REFINANCIACION", "REFINANCIACION_ANTERIOR"]), "credito_id"]).dropna())
c["fue_refinanciado_despues"] = c["cid"].isin(refinanciados_despues)
print("\n", pd.crosstab(c["en_refinanciacion"], c["fue_refinanciado_despues"], margins=True).to_string())
regla("R23", "Columnas de estado posterior a t0 en bd_credito (estado, categoria_credito, en_refinanciacion)", 3, 34,
      "Excluidas como predictores (fuga)")

en_refinanciacion  APROBADA  NO_APLICA
categoria_credito                     
NUEVO                     0       1105
REFINANCIADO            150         12

categoria_inicial: {'NUEVO': 1267}

 fue_refinanciado_despues  False  True   All
en_refinanciacion                          
APROBADA                      0   150   150
NO_APLICA                  1093    24  1117
All                        1093   174  1267


**Lectura.** `en_refinanciacion = APROBADA` coincide casi uno a uno con los créditos que luego fueron refinanciados: es información del futuro y queda fuera. `categoria_inicial` es constante ("NUEVO"), así que no aporta nada. La auditoría completa de fugas, variable por variable, está en el notebook 03.

## 7. Resumen de reglas de calidad

In [17]:
tabla_reglas = pd.DataFrame(reglas)
io_datos.guardar_resultado(tabla_reglas, "fase1", "reglas_calidad")
tabla_reglas[tabla_reglas["afectados"] > 0]

,codigo,regla,afectados,base,pct,accion
15,R01b,Misma persona con más de un id de cliente,2,670,0.30,Unificar el grupo de cliente para la validació...
18,R03,"Pagos ACTIVO casi duplicados (mismo crédito, f...",7,8883,0.08,Se conservan (pueden ser dos cuotas legítimas ...
33,R05,Crédito no anulado sin solicitud enlazada,194,1161,16.71,t0 = fecha inicial del crédito
38,R10,Solicitud posterior al inicio del crédito,26,967,2.69,t0 = mínimo entre solicitud e inicio
41,R13,Desembolso más de 30 días después del inicio,2,1161,0.17,t_ref = primera fecha de desembolso
42,R14,Vencimiento contractual anterior o igual al in...,1,1161,0.09,Revisar con la operación
47,R19,Cuotas del plan DESEMBOLSO distintas de cantid...,1,1160,0.09,Se usa el plan (fechas reales de exigibilidad)
50,R22,Edad en t0 < 18 o > 80 años,3,1161,0.26,Se marca como faltante en el dataset analítico
51,R23,Columnas de estado posterior a t0 en bd_credit...,3,34,8.82,Excluidas como predictores (fuga)


In [18]:
io_datos.escribir_registro("01_auditoria_tablas", {
    "tablas": len(T), "reglas_evaluadas": len(tabla_reglas),
    "reglas_con_hallazgos": int((tabla_reglas["afectados"] > 0).sum()),
    "salidas": ["resultados/fase1/inventario_tablas.csv", "resultados/fase1/duplicados_por_tabla.csv",
                "resultados/fase1/integridad_referencial.csv", "resultados/fase1/rango_fechas.csv",
                "resultados/fase1/reglas_calidad.csv"]})
print(f"{len(tabla_reglas)} reglas evaluadas; {(tabla_reglas['afectados'] > 0).sum()} con al menos un caso.")

52 reglas evaluadas; 9 con al menos un caso.


## Conclusiones de la auditoría de tablas

- **Una persona aparece con dos `id` de cliente.** Es poco, pero rompería la independencia entre pliegues. Por eso todo el agrupamiento de la tesis usa `grupo_cliente`, que unifica los registros con el mismo documento (guardado en `_privado/grupo_cliente_v2.parquet`).
- Los duplicados exactos de `bd_credito` y `bd_solicitud` están solo en créditos anulados y solicitudes rechazadas, que no entran en la población.
- Las llaves primarias son únicas y la integridad referencial de las relaciones críticas para el objetivo (plan, pagos, desembolso) se sostiene. Los conteos exactos de huérfanos están en la tabla de integridad.
- Los duplicados exactos de pagos son la única corrección automática que aplico. Los casi duplicados se conservan por la naturaleza del recaudo por nómina.
- Las inconsistencias de fechas tienen una regla de resolución explícita (t0 = mínimo entre solicitud e inicio; t_ref = primer desembolso), y la misma regla se usa en todo el pipeline.
- Tres columnas de `bd_credito` (`estado`, `categoria_credito`, `en_refinanciacion`) se actualizan después del otorgamiento y quedan vetadas como predictores.